In [ ]:
# Package path adaptation only: keep all legacy scientific calculations unchanged.
from pathlib import Path as _PackagePath
import os as _package_os
_package_candidates = (_PackagePath.cwd(), _PackagePath.cwd().parent, _PackagePath.cwd() / "single_period")
_package_root = next((p for p in _package_candidates if (p / "notebooks").is_dir() and (p / "pinn_results").is_dir()), None)
if _package_root is None:
    raise RuntimeError("Start this notebook from the single_period package or one of its immediate subdirectories.")
_package_os.chdir(_package_root)
(_package_root / "generated" / "one-period-result-fig").mkdir(parents=True, exist_ok=True)
(_package_root / "generated" / "comparison_results").mkdir(parents=True, exist_ok=True)
import numpy as np
from scipy.stats import norm

# 生成标的资产价格路径（适配您的SDE模型）
def generate_paths(A0, N, M, tau, T, beta, alpha, pi):
    """
    生成基于SDE的蒙特卡洛路径：
    dA_t = π * Φ’(Φ^{-1}(A_t/π)) * sqrt(z_t) dW_t, z_t = beta*(T-t)^{-alpha}
    """
    dt = tau / M
    paths = np.zeros((N, M + 1))
    paths[:, 0] = A0  # 初始价格

    for t in range(M):
        current_time = t * dt  # 当前时间
        z_t = beta * (T - current_time) ** (-alpha)  # 计算z_t
        A_current = paths[:, t]  # 当前资产价格
        
        # 计算Φ^{-1}(A_t/π)和Φ’(Φ^{-1}(A_t/π))
        A_pi_ratio = A_current / pi
        A_pi_ratio = np.clip(A_pi_ratio, 1e-6, 1 - 1e-6)  # 防止数值溢出
        Phi_inv = norm.ppf(A_pi_ratio)  # 逆正态CDF
        Phi_prime = norm.pdf(Phi_inv)    # 正态PDF
        
        # 计算波动率项
        sigma_t = pi * Phi_prime * np.sqrt(z_t)
        
        # 生成随机项并更新路径（欧拉离散）
        epsilon = np.random.normal(0, 1, N)
        dA = sigma_t * np.sqrt(dt) * epsilon
        paths[:, t + 1] = A_current + dA
        
        # 防止资产价格为负
        paths[:, t + 1] = np.maximum(paths[:, t + 1], 0.0)
    
    return paths

# LSM方法定价美式期权
def lsm_american_option(paths, K, r, tau, M, option_type='call'):
    """
    LSM定价美式期权（看涨或看跌）
    """
    dt = tau / M
    cash_flows = np.zeros_like(paths)  # 存储每个时间点的现金流

    # 初始化到期日收益
    if option_type == 'call':
        cash_flows[:, -1] = np.maximum(paths[:, -1] - K, 0)
    else:
        cash_flows[:, -1] = np.maximum(K - paths[:, -1], 0)

    # 逆向递归（从到期日前一步开始）
    for t in range(M - 1, 0, -1):
        # 筛选实值路径
        if option_type == 'call':
            in_the_money = paths[:, t] > K
        else:
            in_the_money = paths[:, t] < K
        
        X = paths[in_the_money, t]
        if len(X) == 0:
            continue
        
        # 未来现金流的现值（贴现到当前时间t）
        Y = cash_flows[in_the_money, t + 1] * np.exp(-r * dt)
        
        # 最小二乘回归（二次多项式）
        X_poly = np.column_stack([X, X**2, X**3])
        coef = np.linalg.lstsq(X_poly, Y, rcond=None)[0]
        continuation_value = np.dot(X_poly, coef)
        
        # 立即行权价值
        if option_type == 'call':
            immediate_value = np.maximum(X - K, 0)
        else:
            immediate_value = np.maximum(K - X, 0)
        
        # 更新现金流：若立即行权更优，则替换为当前值
        exercise = immediate_value > continuation_value
        cash_flows[in_the_money, t] = np.where(exercise, immediate_value, Y)
        cash_flows[in_the_money, t + 1:] = 0  # 行权后未来现金流清零

    # 计算期权价格（贴现到初始时刻）
    option_price = np.mean(cash_flows[:, 1:]) * np.exp(-r * dt)
    return option_price

# 参数设置
A0 = 100.0    # 初始资产价格
K = 50.0       # 行权价
r = 0.05       # 无风险利率
tau = 3.0      # 期权到期时间
T = 4.0        # 模型参数T
beta = 0.5     # 模型参数beta
alpha = 0.8    # 模型参数alpha
pi = 100.0     # 模型参数pi
N = 10000      # 路径数量
M = 100        # 时间步数

# 生成路径
paths = generate_paths(A0, N, M, tau, T, beta, alpha, pi)

# 定价美式看涨期权
american_price = lsm_american_option(paths, K, r, tau, M, option_type='call')
print(f"美式看涨期权价格（LSM）: {american_price:.4f}")

# 欧式期权直接贴现到期收益
european_payoff = np.maximum(paths[:, -1] - K, 0)
european_price = np.mean(european_payoff) * np.exp(-r * tau)
print(f"欧式看涨期权价格（蒙特卡洛）: {european_price:.4f}")

In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def generate_paths(A0, N, M, tau, T, beta, alpha, pi):
    """
    生成基于SDE的蒙特卡洛路径：
    dA_t = π * Φ’(Φ^{-1}(A_t/π)) * sqrt(z_t) dW_t, z_t = beta*(T-t)^{-alpha}
    """
    dt = tau / M
    paths = np.zeros((N, M + 1))
    paths[:, 0] = A0  # 初始价格

    for t in range(M):
        current_time = t * dt  # 当前时间
        z_t = beta * (T - current_time) ** (-alpha)  # 计算z_t
        A_current = paths[:, t]  # 当前资产价格
        
        # 计算Φ^{-1}(A_t/π)和Φ’(Φ^{-1}(A_t/π))
        A_pi_ratio = A_current / pi
        A_pi_ratio = np.clip(A_pi_ratio, 1e-6, 1 - 1e-6)  # 防止数值溢出
        Phi_inv = norm.ppf(A_pi_ratio)  # 逆正态CDF
        Phi_prime = norm.pdf(Phi_inv)    # 正态PDF
        
        # 计算波动率项
        sigma_t = pi * Phi_prime * np.sqrt(z_t)
        
        # 生成随机项并更新路径（欧拉离散）
        epsilon = np.random.normal(0, 1, N)
        dA = sigma_t * np.sqrt(dt) * epsilon
        paths[:, t + 1] = A_current + dA
        
        # 防止资产价格为负
        paths[:, t + 1] = np.maximum(paths[:, t + 1], 0.0)
    
    return paths

# 参数设置
A0 = 100.0    # 初始资产价格
K = 50.0       # 行权价
r = 0.05       # 无风险利率
tau = 3.0      # 期权到期时间
T = 4.0        # 模型参数T
beta = 0.5     # 模型参数beta
alpha = 0.8    # 模型参数alpha
pi = 100.0     # 模型参数pi
N = 100000     # 路径数量（增加以提高精度）
M = 100        # 时间步数

# 生成路径
paths = generate_paths(A0, N, M, tau, T, beta, alpha, pi)


def calculate_european_price_matrix(paths, K, r, tau, M, A_grid):
    """
    计算三维（A, t, C）的欧式期权价格矩阵
    - A_grid: 标的资产价格的网格点（例如 np.linspace(0, 150, 50)）
    """
    dt = tau / M
    t_grid = np.linspace(0, tau, M + 1)
    price_matrix = np.zeros((len(A_grid), len(t_grid)))  # 存储期权价格矩阵
    
    # 对每个时间点 t，计算对应价格
    for t_idx, t in enumerate(t_grid):
        # 筛选当前时间 t 的资产价格
        A_current = paths[:, t_idx]
        
        # 计算剩余时间到到期日 T 的贴现因子
        T_remaining = tau - t
        discount_factor = np.exp(-r * T_remaining) if T_remaining > 0 else 1.0
        
        # 计算到期收益（直接取路径终点）
        payoff = np.maximum(paths[:, -1] - K, 0)
        
        # 按当前价格分箱，计算每个 A 网格点的平均收益
        for A_idx, A_val in enumerate(A_grid):
            # 找到接近当前网格点 A_val 的路径
            mask = (A_current >= A_val - 1.0) & (A_current <= A_val + 1.0)
            if np.sum(mask) == 0:
                price_matrix[A_idx, t_idx] = 0.0  # 若无样本，设为0
            else:
                price_matrix[A_idx, t_idx] = discount_factor * np.mean(payoff[mask])
    
    return price_matrix, t_grid, A_grid

# 定义资产价格网格
A_grid = np.linspace(0, 100, 100)  # 覆盖合理价格范围

# 计算三维价格矩阵
price_matrix, t_grid, A_grid = calculate_european_price_matrix(paths, K, r, tau, M, A_grid)

In [ ]:

# 转换为网格矩阵用于绘图
T_grid, A_grid = np.meshgrid(t_grid, A_grid)

# 绘制结果
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# 创建曲面图（过滤零值区域）
surf = ax.plot_surface(A_grid, T_grid, price_matrix, cmap='viridis', edgecolor='none')

# 添加标签和标题
ax.set_xlabel('Asset Price (A)', fontsize=12)
ax.set_ylabel('Time (t)', fontsize=12)
ax.set_zlabel('Option Price (C)', fontsize=12)
ax.set_title('Monte Carlo: European Call Option Price Surface', fontsize=14)

# 添加颜色条
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5)

plt.show()